# ETL Bronze - Curvas de aforo (rating curve) y aforos reales ANA

Lee los JSON crudos subidos por `notebooks_local/ana_rating_curve/download_rating_curves_batch.py`
desde `/Volumes/weather/raw/ana_volume/rating_curves/` y los mergea idempotentemente en
`weather.bronze.ana_rating_curve_segments` y `weather.bronze.ana_discharge_measurements`.
Sin tipado (coherente con el resto de Bronze del proyecto): el tipado y la normalizacion
de unidades (ver docs/rating_curve_discharge_plan.md Seccion 2.2) se resuelven en Silver.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, to_json, struct, current_timestamp

CURVE_SEGMENTS_TABLE = "weather.bronze.ana_rating_curve_segments"
CURVE_SEGMENTS_PATH = "/Volumes/weather/raw/ana_volume/rating_curves/curve_segments/"

CURVE_SEGMENT_COLUMNS = [
    "codigoestacao",
    "Numero_Curva",
    "Periodo_Validade_Inicio",
    "Periodo_Validade_Fim",
    "Coef_a",
    "Coef_h0",
    "Coef_n",
    "Cota_Minima",
    "Cota_Maxima",
    "Tabela_Passo_Cota",
    "Tipo_Curva",
    "Tipo_Equacao",
    "Nivel_Consistencia",
    "Data_Ultima_Alteracao",
]

# _metadata.file_path en vez de input_file_name(): la variante legacy no esta soportada
# en Unity Catalog / serverless (UC_COMMAND_NOT_SUPPORTED).
raw_df = spark.read.option("multiLine", True).json(CURVE_SEGMENTS_PATH)

missing_columns = [c for c in CURVE_SEGMENT_COLUMNS if c not in raw_df.columns]
if missing_columns:
    print(f"Aviso: columnas esperadas ausentes en el JSON crudo (se completan como NULL): {missing_columns}")
    for c in missing_columns:
        raw_df = raw_df.withColumn(c, lit(None).cast("string"))

segments_df = (
    raw_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("raw", to_json(struct([col(c) for c in CURVE_SEGMENT_COLUMNS])))
    .select(*[col(c).cast("string").alias(c) for c in CURVE_SEGMENT_COLUMNS], "raw", "source_file")
    .withColumn("ingested_at", current_timestamp())
    .dropDuplicates(["codigoestacao", "Numero_Curva", "Periodo_Validade_Inicio", "Periodo_Validade_Fim"])
)

delta_table = DeltaTable.forName(spark, CURVE_SEGMENTS_TABLE)
(
    delta_table.alias("t")
    .merge(
        segments_df.alias("s"),
        "t.codigoestacao = s.codigoestacao AND t.Numero_Curva = s.Numero_Curva "
        "AND t.Periodo_Validade_Inicio = s.Periodo_Validade_Inicio AND t.Periodo_Validade_Fim = s.Periodo_Validade_Fim"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
select count(*) as n_segmentos, count(distinct codigoestacao) as n_estaciones from weather.bronze.ana_rating_curve_segments

In [0]:
DISCHARGE_TABLE = "weather.bronze.ana_discharge_measurements"
DISCHARGE_PATH = "/Volumes/weather/raw/ana_volume/rating_curves/discharge_measurements/"

# Los campos crudos de este endpoint vienen con espacios/unidades en el nombre
# ("Cota (cm)", "Vazao (m3/s)"), a diferencia del resto de endpoints ANA del proyecto.
RAW_TO_BRONZE = {
    "codigoestacao": "codigoestacao",
    "Data_Hora_Dado": "Data_Hora_Dado",
    "Cota (cm)": "Cota",
    "Vazao (m3/s)": "Vazao",
}

raw_afo_df = spark.read.option("multiLine", True).json(DISCHARGE_PATH)

missing_raw_columns = [c for c in RAW_TO_BRONZE if c not in raw_afo_df.columns]
if missing_raw_columns:
    print(f"Aviso: columnas esperadas ausentes en el JSON crudo de aforos: {missing_raw_columns}")

present_raw_columns = [c for c in RAW_TO_BRONZE if c in raw_afo_df.columns]

measurements_df = (
    raw_afo_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("raw", to_json(struct([col(c) for c in present_raw_columns])))
    .select(*[col(c).cast("string").alias(RAW_TO_BRONZE[c]) for c in present_raw_columns], "raw", "source_file")
    .withColumn("ingested_at", current_timestamp())
    .dropDuplicates(["codigoestacao", "Data_Hora_Dado"])
)

delta_table_afo = DeltaTable.forName(spark, DISCHARGE_TABLE)
(
    delta_table_afo.alias("t")
    .merge(
        measurements_df.alias("s"),
        "t.codigoestacao = s.codigoestacao AND t.Data_Hora_Dado = s.Data_Hora_Dado"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
%sql
select count(*) as n_aforos, count(distinct codigoestacao) as n_estaciones from weather.bronze.ana_discharge_measurements